In [65]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter
# PLINK v1.90b7 64-bit (16 Jan 2023)
# GERMLINE

In [10]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter

DATA_DIR = Path(".")
GT_TXT = DATA_DIR / "20140625_related_individuals.txt"

def canonical_pair(a: str, b: str):
    return (a, b) if a <= b else (b, a)

def parse_reason(sample_id: str, reason: str):
    reason = str(reason).strip()
    out = []

    # Sibling
    m = re.match(r"^Sibling:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Sibling"))
        return out

    # Parent
    m = re.match(r"^Parent:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Parent"))
        return out

    # Second Order
    m = re.match(r"^Second\s+Order:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "SecondOrder"))
        return out

    # Trio
    m = re.match(r"^trio:(\S+),(\S+)$", reason)
    if m:
        p1, p2 = m.group(1), m.group(2)
        a, b = canonical_pair(sample_id, p1)
        out.append((a, b, "Trio"))
        a, b = canonical_pair(sample_id, p2)
        out.append((a, b, "Trio"))
        return out

    return out


# 1) Read file
df = pd.read_csv(GT_TXT, sep=r"\t+", engine="python")

# 🔥 FIX: strip column whitespace
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# 2) Build pairs
pairs = []
unknown_reasons = Counter()

for _, row in df.iterrows():
    sid = str(row["Sample"]).strip()
    reason = row["Reason for exclusion"]
    extracted = parse_reason(sid, reason)
    if extracted:
        pairs.extend(extracted)
    else:
        unknown_reasons[str(reason).strip()] += 1

pairs_df = (
    pd.DataFrame(pairs, columns=["id1", "id2", "relation"])
    .drop_duplicates()
    .sort_values(["id1", "id2", "relation"])
)

# 3) Write files
pairs_df.to_csv("gt_pairs.tsv", sep="\t", index=False, header=False)

keep_ids = sorted(set(pairs_df["id1"]).union(set(pairs_df["id2"])))
with open("keep.samples", "w") as f:
    for x in keep_ids:
        f.write(x + "\n")

print("gt_pairs.tsv written. n_pairs =", len(pairs_df))
print("keep.samples written. n_samples =", len(keep_ids))
print("\nRelation counts:")
print(pairs_df["relation"].value_counts())

pairs_df.head(10)

Columns: ['Sample', 'Population', 'Gender', 'Reason for exclusion']
gt_pairs.tsv written. n_pairs = 35
keep.samples written. n_samples = 65

Relation counts:
relation
Sibling        11
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64


,id1,id2,relation
0,HG00119,HG00124,SecondOrder
1,HG00501,HG00524,Sibling
2,HG00581,HG00635,Sibling
3,HG00658,HG00702,Sibling
4,HG00731,HG00733,Trio
5,HG00732,HG00733,Trio
6,HG01936,HG01983,SecondOrder
7,HG02024,HG02025,Trio
8,HG02024,HG02026,Trio
9,HG02046,HG02067,Sibling


In [11]:
# ground-truth pairs
def show_pairs_for(sample_id: str, n=20):
    sample_id = sample_id.strip()
    sub = pairs_df[(pairs_df["id1"] == sample_id) | (pairs_df["id2"] == sample_id)]
    return sub.head(n)

show_pairs_for("HG00733")

,id1,id2,relation
4,HG00731,HG00733,Trio
5,HG00732,HG00733,Trio


In [15]:
from pathlib import Path

VCF_FULL = Path("ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz")
VCF_REL  = Path("ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz")
KEEP = Path("keep.samples")

assert VCF_FULL.exists(), VCF_FULL
assert VCF_REL.exists(), VCF_REL
assert KEEP.exists(), KEEP

keep_ids = [x.strip() for x in KEEP.read_text().splitlines() if x.strip()]
print("keep.samples n =", len(keep_ids))


keep.samples n = 65


In [16]:
import subprocess

def vcf_samples(vcf_path: Path):
    out = subprocess.check_output(["bcftools", "query", "-l", str(vcf_path)], text=True)
    return [x.strip() for x in out.splitlines() if x.strip()]

full_ids = set(vcf_samples(VCF_FULL))
rel_ids  = set(vcf_samples(VCF_REL))
keep_set = set(keep_ids)

in_full = sorted(keep_set & full_ids)
in_rel  = sorted(keep_set & rel_ids)
in_both = sorted(keep_set & full_ids & rel_ids)
missing = sorted(keep_set - (full_ids | rel_ids))

print("In FULL:", len(in_full))
print("In REL :", len(in_rel))
print("In BOTH:", len(in_both), "(will take from FULL to avoid duplicates)")
print("Missing:", len(missing))

if missing:
    print("Missing sample IDs (not found in either VCF):")
    print(missing[:20])

In FULL: 33
In REL : 31
In BOTH: 0 (will take from FULL to avoid duplicates)
Missing: 1
Missing sample IDs (not found in either VCF):
['HG00658']


In [19]:
missing = {"HG00658"}  

# 1) keep.present
keep = [x.strip() for x in open("keep.samples") if x.strip()]
keep_present = [x for x in keep if x not in missing]

with open("keep.present", "w") as f:
    for x in keep_present:
        f.write(x + "\n")

print("keep.samples:", len(keep))
print("keep.present:", len(keep_present))
print("removed:", sorted(set(keep) - set(keep_present)))

# 2) gt_pairs.present.tsv
gt = pd.read_csv("gt_pairs.tsv", sep="\t", header=None, names=["id1","id2","relation"])
gt_present = gt[~gt["id1"].isin(missing) & ~gt["id2"].isin(missing)].copy()
gt_present.to_csv("gt_pairs.present.tsv", sep="\t", index=False, header=False)

print("gt_pairs.tsv:", len(gt))
print("gt_pairs.present.tsv:", len(gt_present))
gt_present["relation"].value_counts()

keep.samples: 65
keep.present: 64
removed: ['HG00658']
gt_pairs.tsv: 35
gt_pairs.present.tsv: 34


relation
Sibling        10
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64

In [20]:
from pathlib import Path
import subprocess

VCF_FULL = "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
VCF_REL  = "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz"

def vcf_samples(vcf_path: str):
    out = subprocess.check_output(["bcftools", "query", "-l", vcf_path], text=True)
    return set([x.strip() for x in out.splitlines() if x.strip()])

keep_present_set = set([x.strip() for x in open("keep.present") if x.strip()])
full_ids = vcf_samples(VCF_FULL)
rel_ids  = vcf_samples(VCF_REL)

take_from_full = sorted(keep_present_set & full_ids)
take_from_rel  = sorted(keep_present_set & rel_ids)

Path("keep.present.from_full").write_text("\n".join(take_from_full) + ("\n" if take_from_full else ""))
Path("keep.present.from_rel").write_text("\n".join(take_from_rel) + ("\n" if take_from_rel else ""))

print("Take from FULL:", len(take_from_full))
print("Take from REL :", len(take_from_rel))
print("Union:", len(set(take_from_full) | set(take_from_rel)))
print("Expected keep.present:", len(keep_present_set))

Take from FULL: 33
Take from REL : 31
Union: 64
Expected keep.present: 64


In [21]:
OUT_FULL = "gt_only.present.from_full.vcf.gz"
OUT_REL  = "gt_only.present.from_rel.vcf.gz"
OUT_MERGED = "gt_only.present.vcf.gz"

# subset FULL
!bcftools view -S keep.present.from_full -Oz -o {OUT_FULL} {VCF_FULL}
!bcftools index -t {OUT_FULL}
print("Subset FULL Completed")

# subset REL
!bcftools view -S keep.present.from_rel -Oz -o {OUT_REL} {VCF_REL}
!bcftools index -t {OUT_REL}
print("Subset REL Completed")

# merge (samples are disjoint)
!bcftools merge -Oz -o {OUT_MERGED} {OUT_FULL} {OUT_REL}
!bcftools index -t {OUT_MERGED}
print("Merge Completed")

# sanity: sample count
!echo "keep.present:" && wc -l keep.present
!echo "gt_only.present.vcf.gz samples:" && bcftools query -l gt_only.present.vcf.gz | wc -l

keep.present:
64 keep.present
gt_only.present.vcf.gz samples:
64


In [22]:
out_ids = subprocess.check_output(["bcftools", "query", "-l", "gt_only.present.vcf.gz"], text=True).splitlines()
out_ids = [x.strip() for x in out_ids if x.strip()]

assert len(out_ids) == len(set(out_ids)), "Duplicate sample IDs in merged VCF!"
assert set(out_ids) == keep_present_set, "Output sample set != keep.present!"

print("gt_only.present.vcf.gz is consistent:")
print("  n_samples =", len(out_ids))
print("  no duplicates, matches keep.present")

gt_only.present.vcf.gz is consistent:
  n_samples = 64
  no duplicates, matches keep.present


In [27]:
!source /home/jovyan/.bashrc

In [35]:
# directly use plink
!plink \
  --bfile gt_only.present \
  --geno 0.02 \
  --mind 0.02 \
  --maf 0.05 \
  --make-bed \
  --out gt_only.present.qc

!plink \
  --bfile gt_only.present.qc \
  --indep-pairwise 50 5 0.2 \
  --out gt_only.present.qc.prune

!plink \
  --bfile gt_only.present.qc \
  --extract gt_only.present.qc.prune.prune.in \
  --make-bed \
  --out gt_only.present.qc.pruned

# the result have extremely high P_HAT value for most of the individuals, so we decided to prune first

/bin/bash: line 1: plink: command not found
/bin/bash: line 1: plink: command not found
/bin/bash: line 1: plink: command not found


In [36]:
# prune before plink
'''
--geno 0.02：去掉缺失率 >2% 的 SNP
--mind 0.02：去掉缺失率 >2% 的样本（样本很少，通常不会删很多）
--maf 0.05：去掉超稀有 SNP（稀有位点会让 IBD 更不稳）
'''
!plink \
  --bfile gt_only.present \
  --geno 0.02 \
  --mind 0.02 \
  --maf 0.05 \
  --make-bed \
  --out gt_only.present.qc

!plink \
  --bfile gt_only.present.qc \
  --indep-pairwise 50 5 0.2 \
  --out gt_only.present.qc.prune

!plink \
  --bfile gt_only.present.qc \
  --extract gt_only.present.qc.prune.prune.in \
  --make-bed \
  --out gt_only.present.qc.pruned

!plink \
  --bfile gt_only.present.qc.pruned \
  --genome full \
  --out plink_genome.present.qcpruned

/bin/bash: line 1: plink: command not found
/bin/bash: line 1: plink: command not found
/bin/bash: line 1: plink: command not found
/bin/bash: line 1: plink: command not found


/bin/bash: line 1: plink: command not found


In [37]:
gt = pd.read_csv("gt_pairs.present.tsv", sep="\t", header=None, names=["id1","id2","relation"])
gt_pairs = set(zip(gt["id1"].astype(str), gt["id2"].astype(str)))
K = len(gt_pairs)

print("GT present pairs =", K)
gt["relation"].value_counts()

GT present pairs = 34


relation
Sibling        10
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64

In [38]:
gen = pd.read_csv("plink_genome.present.qcpruned.genome", sep=r"\s+", engine="python")
gen.head()

,FID1,IID1,FID2,IID2,RT,EZ,Z0,Z1,Z2,PI_HAT,PHE,DST,PPC,RATIO,IBS0,IBS1,IBS2,HOMHOM,HETHET
0,HG00119,HG00119,HG00524,HG00524,UN,NaN,1.0000,0.0000,0.0000,0.0000,-1,0.723651,0.1763,1.5926,9040,41257,57062,27.0,43.0
1,HG00119,HG00119,HG00581,HG00581,UN,NaN,1.0000,0.0000,0.0000,0.0000,-1,0.726993,0.0455,1.3333,9091,40437,57830,30.0,40.0
2,HG00119,HG00119,HG00731,HG00731,UN,NaN,0.8793,0.0340,0.0867,0.1037,-1,0.765099,0.8465,2.6316,5678,39080,62598,19.0,50.0
3,HG00119,HG00119,HG00732,HG00732,UN,NaN,0.8810,0.0379,0.0811,0.1000,-1,0.764026,0.6323,2.1818,5689,39289,62379,22.0,48.0
4,HG00119,HG00119,HG01936,HG01936,UN,NaN,1.0000,0.0000,0.0000,0.0000,-1,0.728001,0.5000,2.0000,9553,39295,58507,23.0,46.0


In [58]:
# use plink (z1+z2) to find relationship (First/second degree)
gen = pd.read_csv("plink_genome.present.qcpruned.genome", sep=r"\s+", engine="python")

gen["Z1Z2"] = gen["Z1"] + gen["Z2"]

# gen[["IID1","IID2","Z0","Z1","Z2","PI_HAT","Z1Z2"]].head()

def canonical_pair(a, b):
    return (a, b) if a <= b else (b, a)

# PC_thres = 0.95
# sib_thres = 0.70
# second_thres = 0.45

First_thres = 0.7
Second_thres = 0.4

rows = []
for _, r in gen.iterrows():
    score = float(r["Z1Z2"])
    if score >= First_thres:
        rel = "First"
    elif score >= Second_thres:
        rel = "Second"
    # elif score >= TH_SECOND:
    #     rel = "SecondOrder"
    else:
        continue
    
    a, b = canonical_pair(str(r["IID1"]), str(r["IID2"]))
    rows.append((a, b, rel))

plink_z1z2 = (
    pd.DataFrame(rows, columns=["id1","id2","relation"])
    .drop_duplicates()
    .sort_values(["id1","id2"])
)

plink_z1z2.to_csv("plink_z1z2.present.tsv", sep="\t", index=False, header=False)

print("wrote plink_z1z2.present.tsv")
print("n_pred =", len(plink_z1z2))
plink_z1z2, plink_z1z2.relation.value_counts()

wrote plink_z1z2.present.tsv
n_pred = 29


(        id1      id2 relation
 0   HG00119  HG00124   Second
 1   HG00581  HG00635    First
 2   HG00731  HG00733    First
 3   HG00732  HG00733    First
 4   HG01936  HG01983    First
 5   HG02024  HG02025    First
 6   HG02024  HG02026    First
 7   HG02046  HG02067   Second
 8   HG02250  HG02377    First
 9   HG02371  HG02372    First
 10  HG02373  HG02381    First
 11  HG02375  HG02388   Second
 27  HG02381  HG02388   Second
 12  HG02386  HG02387   Second
 13  HG03673  HG03948    First
 14  HG03713  HG03715    First
 15  NA19238  NA19240    First
 16  NA19239  NA19240    First
 17  NA19313  NA19331    First
 28  NA19660  NA19685    First
 18  NA19661  NA19685    First
 19  NA19675  NA19678    First
 20  NA19675  NA19679    First
 21  NA20289  NA20341    First
 22  NA20334  NA20336   Second
 23  NA20526  NA20792    First
 24  NA20868  NA20871    First
 25  NA20886  NA20898    First
 26  NA20893  NA20895   Second,
 relation
 First     22
 Second     7
 Name: count, dtype: int64)

In [ ]:
# use germline
# 删除缺失的位点
!plink \
  --bfile gt_only.present.qc.pruned \
  --geno 0 \
  --snps-only just-acgt \
  --biallelic-only strict \
  --make-bed \
  --out gt_only.present.qc.pruned.nomissSNP

!plink \
  --bfile gt_only.present.qc.pruned.nomissSNP \
  --recode \
  --out gt_only.present.qcpruned.nomissSNP_ped

# 这里minimum m设置为3的时候，下游match一共只有8条，特别特别特别少
# minimum m = 1
!germline \
  -input gt_only.present.qcpruned.nomissSNP_ped.ped gt_only.present.qcpruned.nomissSNP_ped.map \
  -output germline.present \
  -bits 128 \
  -min_m 1

In [74]:
MAP = "gt_only.present.qcpruned.nomissSNP_ped.map"
m = pd.read_csv(MAP, sep=r"\s+", header=None, engine="python")

bp = m[3].astype(int)

min_bp = bp.min()
max_bp = bp.max()
L_bp = max_bp - min_bp + 1
L_mb = L_bp / 1e6

MATCH = "germline.present.match"
GT = "gt_pairs.present.tsv"

print("Using:")
print("  MATCH =", MATCH)
print("  GT    =", GT)
print("  L_mb  =", L_mb)

Using:
  MATCH = germline.present.match
  GT    = gt_pairs.present.tsv
  L_mb  = 35.189572


In [78]:
df = pd.read_csv(MATCH, sep=r"\s+", header=None, engine="python")

# match columns based on your file:
# 0=id1, 2=id2, 10=seg_len, 11=unit(MB)
print("Units:", df[11].astype(str).unique())

id1 = df[0].astype(str)
id2 = df[2].astype(str)
seg_len_mb = df[10].astype(float)

def canon(a, b):
    return (a, b) if a <= b else (b, a)

pairs = [canon(a, b) for a, b in zip(id1, id2)]

seg = pd.DataFrame(pairs, columns=["id1","id2"])
seg["seg_len_mb"] = seg_len_mb

agg = (seg.groupby(["id1","id2"])
          .agg(total_ibd_mb=("seg_len_mb","sum"),
               n_seg=("seg_len_mb","size"),
               max_seg_mb=("seg_len_mb","max"))
          .reset_index())

agg["z1z2_hat"] = agg["total_ibd_mb"] / L_mb

agg = agg.sort_values(["z1z2_hat","total_ibd_mb","n_seg"], ascending=False)

agg.to_csv("germline_pairs_norm.present.tsv", sep="\t", index=False)
print("wrote germline_pairs_norm.present.tsv, n_pairs =", len(agg))
agg.head(15)

Units: ['MB']
wrote germline_pairs_norm.present.tsv, n_pairs = 17


,id1,id2,total_ibd_mb,n_seg,max_seg_mb,z1z2_hat
7,HG02046,HG02067,22.814,4,7.915,0.648317
16,NA20526,NA20792,22.735,7,12.910,0.646072
0,HG00581,HG00635,8.126,3,5.008,0.230921
15,NA20334,NA20336,5.231,2,3.839,0.148652
9,HG02373,HG02381,3.849,2,2.401,0.109379
10,HG02375,HG02388,2.553,2,1.295,0.072550
12,NA19238,NA20344,1.326,1,1.326,0.037682
14,NA20289,NA20341,1.234,1,1.234,0.035067
8,HG02353,HG02372,1.092,1,1.092,0.031032
6,HG00732,NA19660,1.084,1,1.084,0.030805


In [80]:
TH_FIRST = 0.25
TH_SECOND = 0.10

def classify(z):
    if z >= TH_FIRST:
        return "FirstDegree_like"
    elif z >= TH_SECOND:
        return "SecondDegree_like"
    else:
        return "Distant_or_Unrelated"

pred = agg.copy()
pred["relation"] = pred["z1z2_hat"].apply(classify)

print(pred["relation"].value_counts())
pred.head(20)

relation
Distant_or_Unrelated    12
SecondDegree_like        3
FirstDegree_like         2
Name: count, dtype: int64


,id1,id2,total_ibd_mb,n_seg,max_seg_mb,z1z2_hat,relation
7,HG02046,HG02067,22.814,4,7.915,0.648317,FirstDegree_like
16,NA20526,NA20792,22.735,7,12.910,0.646072,FirstDegree_like
0,HG00581,HG00635,8.126,3,5.008,0.230921,SecondDegree_like
15,NA20334,NA20336,5.231,2,3.839,0.148652,SecondDegree_like
9,HG02373,HG02381,3.849,2,2.401,0.109379,SecondDegree_like
10,HG02375,HG02388,2.553,2,1.295,0.072550,Distant_or_Unrelated
12,NA19238,NA20344,1.326,1,1.326,0.037682,Distant_or_Unrelated
14,NA20289,NA20341,1.234,1,1.234,0.035067,Distant_or_Unrelated
8,HG02353,HG02372,1.092,1,1.092,0.031032,Distant_or_Unrelated
6,HG00732,NA19660,1.084,1,1.084,0.030805,Distant_or_Unrelated


In [79]:
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,HG00581,HG00581,HG00635,HG00635,22,17303596,18547016,.,.,4224,1.243,MB,0,0,0
1,NA20792,NA20792,NA20526,NA20526,22,17406302,18642544,.,.,4096,1.236,MB,0,0,0
2,NA20334,NA20334,NA20336,NA20336,22,18875507,20267689,.,.,4224,1.392,MB,0,0,0
3,HG02373,HG02373,HG02381,HG02381,22,18903537,20351819,.,.,4224,1.448,MB,1,0,0
4,NA20792,NA20792,NA20526,NA20526,22,18875507,20351819,.,.,4352,1.476,MB,0,0,0
5,HG02373,HG02373,HG02381,HG02381,22,21889545,24290680,.,.,8576,2.401,MB,0,0,0
6,NA20792,NA20792,NA20526,NA20526,22,23669316,25014440,.,.,3712,1.345,MB,0,0,0
7,NA20334,NA20334,NA20336,NA20336,22,23690725,27530184,.,.,11008,3.839,MB,0,0,0
8,HG00731,HG00731,HG00733,HG00733,22,28170690,29233924,.,.,1152,1.063,MB,0,0,0
9,HG02353,HG02353,HG02372,HG02372,22,28312557,29404706,.,.,1280,1.092,MB,0,0,0
